In [1]:
"""
Top 3 Categories Per Score  +  Top 3 Mechanics Per Score
=========================================================
Two transparent-background charts showing the most distinctively-represented
categories / mechanics in each rounded score bucket (1-10), ranked by lift
(rate-in-bucket / rate-in-corpus). Designed to drop into a dark slide deck.

Outputs:
    eda_charts/M19_top3_categories_per_score.png
    eda_charts/M20_top3_mechanics_per_score.png

Adjust CSV_PATH below if your file lives elsewhere.
"""

import os
import re
import warnings
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# ----------------------------------------------------------------------
# 1. Load + dedupe (the dataset has ~5,100 oversampled duplicates)
# ----------------------------------------------------------------------
CSV_PATH = 'ttrpg_bgg_encoded_dataset.csv'   # change if needed
os.makedirs('eda_charts', exist_ok=True)

df = pd.read_csv(CSV_PATH).drop_duplicates().reset_index(drop=True)
df['ord10'] = df['Average Score'].round().astype(int).clip(1, 10)

cat_cols = [c for c in df.columns if c.startswith('Cat_')]
mech_cols_raw = [c for c in df.columns if c.startswith('Mech_')]

# ----------------------------------------------------------------------
# 2. Filter out CSV-parsing artifact mechanics
#    (the original "Accessory (dice, cards, maps, ...)" cell got split
#     across columns when the CSV was generated)
# ----------------------------------------------------------------------
def is_clean_mech(c):
    name = c.replace('Mech_', '')
    if name.endswith(')') and '(' not in name: return False
    if name.endswith(':') or name.startswith(' '): return False
    if '(' in name and ')' not in name: return False
    bad = {'cards)', 'etc)', 'char sheets', 'maps', 'screens',
           'Accessory (dice', 'Software (for maps'}
    return name not in bad

mech_cols = [c for c in mech_cols_raw if is_clean_mech(c)]

# ----------------------------------------------------------------------
# 3. Smart label shortener: prefer parenthetical subgenre as the label
#    so 'Fantasy' and 'Fantasy (High Fantasy)' don't both render as 'Fantasy'
# ----------------------------------------------------------------------
SHORTEN = {
    'Roll / Spin and Move': 'Roll & Move',
    'Roll/Spin and Move': 'Roll & Move',
    'Pattern Recognition': 'Pattern Recog.',
    'Pattern Building': 'Pattern Build',
    'Network and Route Building': 'Network/Route',
    'Secret Unit Deployment': 'Secret Deploy',
    'Worker Placement': 'Worker Place',
    'Solo / Solitaire Game': 'Solo Play',
    'Solo/Solitaire Game': 'Solo Play',
    'Variable Set-up': 'Variable Setup',
    'Scenario / Mission / Campaign Game': 'Scenario/Mission',
    'Scenario/Mission/Campaign Game': 'Scenario/Mission',
    'Simultaneous Action Selection': 'Sim. Action Sel.',
    'Action / Dexterity': 'Action/Dex',
    'Comedy / Satire': 'Comedy/Satire',
    'Action / Adventure': 'Action/Adv',
    "Children's Game": "Children's",
    'Generic / Universal': 'Generic/Univ',
    'Generic/Universal': 'Generic/Univ',
    'Movies / TV / Radio theme': 'Movies/TV',
}

def short_name(col, prefix):
    name = col.replace(prefix, '')
    paren = re.search(r'\(([^)]*)\)', name)
    if paren:
        inside = paren.group(1).strip()
        # Drop mechanic-description parens like "(rules/options to enhance play)"
        if any(kw in inside.lower() for kw in
               ['rules', 'options', 'min needed', 'game world',
                'item will', 'rankings', 'rpg related']):
            name = re.sub(r'\s*\(.*?\)', '', name).strip()
        else:
            # Use the parenthetical content as the distinguishing label
            name = inside
    if name in SHORTEN:
        return SHORTEN[name]
    if len(name) > 15:
        name = name[:14] + '..'
    return name

# ----------------------------------------------------------------------
# 4. Per-score top-3 distinctive features (lift-based)
# ----------------------------------------------------------------------
def top_distinctive_features(score, feature_cols, prefix,
                             k=3, min_corpus=40, min_bucket=3):
    """For a given score bucket, return the top-k features whose
    rate-in-bucket / rate-in-corpus (lift) is highest."""
    sub = df[df['ord10'] == score]
    n_bucket, n_corpus = len(sub), len(df)
    if n_bucket == 0:
        return []

    rows = []
    for c in feature_cols:
        corpus_count = df[c].sum()
        if corpus_count < min_corpus:
            continue
        bucket_count = sub[c].sum()
        if bucket_count < min_bucket:
            continue
        lift = (bucket_count / n_bucket) / (corpus_count / n_corpus)
        rows.append((short_name(c, prefix), lift, int(bucket_count)))

    # If two columns shorten to the same label, keep the higher-lift one
    seen = {}
    for name, lift, bc in sorted(rows, key=lambda x: -x[1]):
        seen.setdefault(name, (name, lift, bc))
    out = sorted(seen.values(), key=lambda x: -x[1])[:k]
    return out

# ----------------------------------------------------------------------
# 5. Chart builder (transparent bg, dark-slide friendly)
# ----------------------------------------------------------------------
def build_chart(feature_cols, prefix, cmap, title, subtitle, outfile):
    TEXT, SUBTLE = '#FFFFFF', '#D1D5DB'
    plt.rcParams['font.family'] = 'DejaVu Sans'

    fig, axes = plt.subplots(2, 5, figsize=(24, 12))
    fig.patch.set_alpha(0)

    for i, score in enumerate(range(1, 11)):
        ax = axes.flat[i]
        ax.patch.set_alpha(0)
        color = cmap((score - 1) / 9)

        n = (df['ord10'] == score).sum()
        items = top_distinctive_features(score, feature_cols, prefix)

        if not items:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                    color=TEXT, fontsize=14, transform=ax.transAxes)
            ax.set_xticks([]); ax.set_yticks([])
            for s in ax.spines.values(): s.set_visible(False)
            ax.set_title(f'Score {score}', fontsize=22, fontweight='bold',
                         color=TEXT, pad=16, loc='left')
            continue

        names = [w for w, _, _ in items]
        lifts = [l for _, l, _ in items]

        bars = ax.bar(names, lifts, color=color, edgecolor='none', width=0.62)

        for bar, l in zip(bars, lifts):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + max(lifts) * 0.04,
                    f'{l:.1f}x', ha='center', va='bottom',
                    fontsize=15, color=TEXT, fontweight='bold')

        ax.set_title(f'Score {score}', fontsize=22, fontweight='bold',
                     color=TEXT, pad=16, loc='left')
        ax.text(0.0, 1.02, f'n={n:,}', transform=ax.transAxes,
                fontsize=12, color=SUBTLE, ha='left', va='bottom')

        ax.tick_params(axis='x', colors=TEXT, labelsize=12, pad=8, length=0)
        ax.tick_params(axis='y', length=0, labelsize=0, colors='none')
        for label in ax.get_xticklabels():
            label.set_fontweight('bold')
            label.set_color(TEXT)
            label.set_rotation(20)
            label.set_ha('right')

        for s in ax.spines.values(): s.set_visible(False)
        ax.grid(False)
        ax.set_yticks([])
        ax.set_ylim(0, max(lifts) * 1.32)

    fig.suptitle(title, fontsize=30, fontweight='bold',
                 color=TEXT, x=0.02, y=0.99, ha='left')
    fig.text(0.02, 0.945, subtitle, fontsize=14, color=SUBTLE, ha='left')

    plt.tight_layout(rect=[0.01, 0.02, 0.99, 0.91], h_pad=5.5, w_pad=2.5)
    plt.savefig(f'eda_charts/{outfile}', dpi=300, bbox_inches='tight',
                facecolor='none', edgecolor='none', transparent=True)
    plt.close()
    print(f'Saved: eda_charts/{outfile}')

# ----------------------------------------------------------------------
# 6. Build both charts
# ----------------------------------------------------------------------
build_chart(
    cat_cols, 'Cat_', plt.cm.RdYlGn,
    title='Top 3 Categories Per Score',
    subtitle=('For each score bucket, the 3 most distinctively-represented '
              'categories (by lift vs full corpus). Color: low score (red / '
              'Flop) -> high score (green / Hit).'),
    outfile='M19_top3_categories_per_score.png',
)

build_chart(
    mech_cols, 'Mech_', plt.cm.plasma,
    title='Top 3 Mechanics Per Score',
    subtitle=('For each score bucket, the 3 most distinctively-represented '
              'mechanics (by lift vs full corpus). Color: plasma scale across '
              'score buckets.'),
    outfile='M20_top3_mechanics_per_score.png',
)

FileNotFoundError: [Errno 2] No such file or directory: 'ttrpg_bgg_encoded_dataset.csv'

In [ ]:
"""
Model Performance Comparison — Bar Chart
=========================================
Dark-theme, transparent-background bar chart comparing baseline RMSE
against trained models. Each bar shows RMSE, % improvement, and R²,
with a red→green RdYlGn gradient running from worst (baseline) to
best (GBM).

Output: eda_charts/M21_model_performance.png

Update the `data` list below to refit your latest results.
"""

import os
import matplotlib.pyplot as plt
import numpy as np

os.makedirs('eda_charts', exist_ok=True)

# ----------------------------------------------------------------------
# 1. Results table — edit this when numbers change
#    Order matters: worst (baseline) -> best, so the red->green gradient
#    runs the same direction as the performance ranking.
# ----------------------------------------------------------------------
data = [
    {'name': 'Baseline\n(Random)',            'rmse': 2.71,  'pct': None,   'r2': None,  'is_baseline': True},
    {'name': 'kNN\n(n=3)',                    'rmse': 0.977, 'pct': 179.00, 'r2': 0.874, 'is_baseline': False},
    {'name': 'Lasso\n(α=0.0001)',             'rmse': 0.950, 'pct': 186.70, 'r2': 0.880, 'is_baseline': False},
    {'name': 'Ridge\n(α=0.75)',               'rmse': 0.942, 'pct': 189.08, 'r2': 0.882, 'is_baseline': False},
    {'name': 'GBM\n(depth=5, n=200, lr=0.1)', 'rmse': 0.79,  'pct': 244.21, 'r2': 0.92,  'is_baseline': False},
]

# ----------------------------------------------------------------------
# 2. Style — matches the M16-M20 chart family
# ----------------------------------------------------------------------
TEXT   = '#FFFFFF'
SUBTLE = '#D1D5DB'
ACCENT = '#86EFAC'   # mint green for the % improvement number
plt.rcParams['font.family'] = 'DejaVu Sans'

# Red (worst) -> green (best) gradient across all bars
cmap = plt.cm.RdYlGn
n = len(data)
colors = [cmap(i / (n - 1)) for i in range(n)]

# ----------------------------------------------------------------------
# 3. Plot
# ----------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(15, 9))
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)

x_pos = np.arange(n)
rmse_values = [d['rmse'] for d in data]
bars = ax.bar(x_pos, rmse_values, color=colors, edgecolor='none', width=0.6)

# Baseline reference line + label
baseline_rmse = data[0]['rmse']
ax.axhline(baseline_rmse, color=SUBTLE, linestyle='--', linewidth=1.3, alpha=0.55)
ax.text(n - 0.55, baseline_rmse + 0.04, f'Baseline RMSE = {baseline_rmse}',
        color=SUBTLE, fontsize=11, ha='right', style='italic')

# Per-bar annotations (stacked: %improvement / RMSE / R²)
for bar, d in zip(bars, data):
    h = bar.get_height()
    x = bar.get_x() + bar.get_width() / 2

    if d['is_baseline']:
        ax.text(x, h + 0.08, f'RMSE  {d["rmse"]:.2f}',
                ha='center', va='bottom',
                fontsize=15, color=TEXT, fontweight='bold')
    else:
        ax.text(x, h + 0.65, f'↓ {d["pct"]:.2f}%',
                ha='center', va='bottom',
                fontsize=18, color=ACCENT, fontweight='bold')
        ax.text(x, h + 0.30, f'RMSE  {d["rmse"]:.3f}',
                ha='center', va='bottom',
                fontsize=14, color=TEXT, fontweight='bold')
        ax.text(x, h + 0.06, f'R² = {d["r2"]:.3f}',
                ha='center', va='bottom',
                fontsize=12, color=SUBTLE)

# X-axis (model names)
ax.set_xticks(x_pos)
ax.set_xticklabels([d['name'] for d in data])
ax.tick_params(axis='x', colors=TEXT, labelsize=13, pad=10, length=0)
for label in ax.get_xticklabels():
    label.set_fontweight('bold')

# Y-axis (RMSE)
ax.set_ylabel('RMSE (lower is better)', color=TEXT, fontsize=13,
              fontweight='bold', labelpad=10)
ax.tick_params(axis='y', colors=SUBTLE, labelsize=10, length=0)
ax.set_ylim(0, max(rmse_values) * 1.35)

# Strip spines, keep a subtle horizontal grid
for s in ax.spines.values():
    s.set_visible(False)
ax.grid(axis='y', linestyle='--', linewidth=0.4, color=SUBTLE, alpha=0.25)
ax.set_axisbelow(True)

# Title block
fig.suptitle('Model Performance Comparison',
             fontsize=28, fontweight='bold', color=TEXT,
             x=0.04, y=0.98, ha='left')
fig.text(0.04, 0.928,
         '10k balanced dataset, all features. Lower RMSE = better fit. '
         'Color gradient: red (baseline) → green (best model).',
         fontsize=13, color=SUBTLE, ha='left')

# Footnote
fig.text(0.04, 0.02,
         'Note: GBM hyperparameters carried over from the description-only '
         'model — not separately tuned.',
         fontsize=10, color=SUBTLE, style='italic', ha='left')

plt.tight_layout(rect=[0.04, 0.06, 0.98, 0.90])
plt.savefig('eda_charts/M21_model_performance.png', dpi=300,
            bbox_inches='tight', facecolor='none', edgecolor='none',
            transparent=True)
plt.close()
print('Saved: eda_charts/M21_model_performance.png')